Este projeto tem como objetivo prever a avaliação que um utilizador atribuiria a um determinado filme, com base no histórico de avaliações do dataset MovieLens. Para isso, serão comparadas diferentes configurações de um sistema de filtragem colaborativa.

### 1 - Carregamento e Exploração de Dados

In [2]:
import pandas as pd

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

display(ratings.head())
display(movies.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
print("Numero de avaliacoes:",len(ratings))
print("Numero de utilizadores:",ratings["userId"].nunique())
print("Numero de filmes no catalogo:",len(movies))

Numero de avaliacoes: 100836
Numero de utilizadores: 610
Numero de filmes no catalogo: 9742


In [4]:
print("Valores em falta nas avaliacoes:\n")
print(ratings.isna().sum())

Valores em falta nas avaliacoes:

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### 2- Para maior facilidade visual, juntar-se-ão as avaliações aos títulos


In [5]:
avaliacoes = ratings.merge(
    movies, 
    on="movieId",
    how="left",
    validate="many_to_one"
)

display(avaliacoes[["userId","title","rating"]].head(10))

,userId,title,rating
0,1,Toy Story (1995),4.0
1,1,Grumpier Old Men (1995),4.0
2,1,Heat (1995),4.0
3,1,Seven (a.k.a. Se7en) (1995),5.0
4,1,"Usual Suspects, The (1995)",5.0
5,1,From Dusk Till Dawn (1996),3.0
6,1,Bottle Rocket (1996),5.0
7,1,Braveheart (1995),4.0
8,1,Rob Roy (1995),5.0
9,1,Canadian Bacon (1995),5.0


Apresentam-se as primeiras dez avaliações, associadas aos respetivos títulos dos filmes. Neste conjunto de dados, estas linhas correspondem ao utilizador 1.

**Observação**: o .head(x) anda a buscar os "x" primeiros dados do DataFrame Pandas.

~*Analizando cenários*~ 
Para caso queira se explorar aos filmes correspondentes do userId 1:

In [6]:
avaliacoes_user1 = avaliacoes.loc[
    avaliacoes["userId"] == 1,
    ["title", "rating"]
]

print("Número de filmes avaliados pelo utilizador 1:", len(avaliacoes_user1))
display(avaliacoes_user1.head(10))

Número de filmes avaliados pelo utilizador 1: 232


,title,rating
0,Toy Story (1995),4.0
1,Grumpier Old Men (1995),4.0
2,Heat (1995),4.0
3,Seven (a.k.a. Se7en) (1995),5.0
4,"Usual Suspects, The (1995)",5.0
5,From Dusk Till Dawn (1996),3.0
6,Bottle Rocket (1996),5.0
7,Braveheart (1995),4.0
8,Rob Roy (1995),5.0
9,Canadian Bacon (1995),5.0


### 3- Nosso objectivo é: dado um filme e um utilizador, uma previsão da nota. Serão utilizados dois modelos de Regressão: **Ridge** e **Árvore de Decisão**.

In [7]:
from sklearn.model_selection import train_test_split

# Agora, ao invés de ir tratar o id como números, converteremos para string, para que possa ser categorizado.

X = ratings[["userId","movieId"]].astype(str)

# Resposta que queremos prever

y= ratings["rating"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Avaliações de treino:",len(X_treino))
print("Avaliações de teste:",len(X_teste))

Avaliações de treino: 80668
Avaliações de teste: 20168


### 4 - Preparação das Categorias e Criação dos Modelos

As variáveis de entrada são os identificadores do utilizador (*userId*) e do filme (*movieId*).

Apesar de se tratar de números, representam as categorias. Os números por si só, não trazem nenhum valor. 

Por isso, os identificadores serão transformados através do ***One-Hot-Encoding** antes de serem utilizados por modelos seguintes.

***One-Hot-Encoding**

As categorias agora, serão transformadas em colunas de "sim" (1) ou "não" (0).

Exemplo:

Os identificadores devem de ser representados como categorias e não quantidades:

*Antes*:

|userId | movieId|
|-------|--------|
|1      |     10|
|2      |    20 |
|1      |     20|

*Depois do **One-Hot-Encoding***:



| É o utilizador 1? | É o utilizador 2? | É o filme 10? | É o filme 20? |
|---|---|---|---|
| 1 | 0 | 1 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 |

*Exemplos ilustrativos, apenas*

Evita confusão de interpretar identificadores como quantidades.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

exemplo = X_treino[["userId","movieId"]].head(5).copy()
codificador_exemplo = OneHotEncoder(handle_unknown="ignore")

exemplo_transformado = codificador_exemplo.fit_transform(exemplo)

#trazer em tabela com *pandas*

tabela_exemplo = pd.DataFrame(
    exemplo_transformado.toarray(), columns=codificador_exemplo.get_feature_names_out(), index=exemplo.index
)

print("Identificadores antes:")
display(exemplo)

print("Com One-Hot-Encoding:")
display(tabela_exemplo)